In [1]:
# COMPUTATIONAL ENVIRONMENT
# HydroFlowNet-XF — Separate Latency Benchmark

import os
import sys
import time
import platform
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("=" * 75)
print("HYDROFLOWNET-XF — COMPUTATIONAL LATENCY BENCHMARK")
print("=" * 75)

def get_cpu_name():
    try:
        with open("/proc/cpuinfo", "r") as f:
            for line in f:
                if "model name" in line:
                    return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "Unknown CPU"

cpu_name = get_cpu_name()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
cuda_version = torch.version.cuda if torch.cuda.is_available() else "N/A"

environment = {
    "Python": sys.version.split()[0],
    "OS": platform.platform(),
    "CPU": cpu_name,
    "PyTorch": torch.__version__,
    "CUDA": cuda_version,
    "GPU": gpu_name,
    "CUDA_available": torch.cuda.is_available(),
    "PyTorch_threads": torch.get_num_threads()
}

for key, value in environment.items():
    print(f"{key:20s}: {value}")

print("=" * 75)
print("✔ Environment information recorded.")

HYDROFLOWNET-XF — COMPUTATIONAL LATENCY BENCHMARK
Python              : 3.13.15
OS                  : Linux-6.6.122+-x86_64-with-glibc2.39
CPU                 : Intel(R) Xeon(R) CPU @ 2.20GHz
PyTorch             : 2.11.0+cpu
CUDA                : N/A
GPU                 : N/A
CUDA_available      : False
PyTorch_threads     : 1
✔ Environment information recorded.


In [2]:
# EXACT FROZEN ARCHITECTURES
# No training is performed in this notebook.

NUM_ZONES = 4
HISTORY_K = 3
LATENT_DIM = 256

class HydroFlowNet_XF(nn.Module):
    def __init__(self, num_zones=NUM_ZONES, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.motion_bn = nn.BatchNorm2d(4)
        self.sensor_bn = nn.BatchNorm2d(3)
        self.motion_enc = nn.Linear(4, latent_dim // 2)
        self.sensor_enc = nn.Linear(3, latent_dim // 2)
        self.gmf_gate = nn.Sequential(
            nn.Linear(latent_dim, 1),
            nn.Sigmoid()
        )
        self.zone_embedding = nn.Embedding(
            num_zones,
            latent_dim // 2
        )
        self.lfie = nn.Conv1d(
            latent_dim,
            latent_dim,
            kernel_size=3,
            padding=1,
            groups=latent_dim
        )
        self.ats = nn.Sequential(
            nn.Linear(1, 1),
            nn.Sigmoid()
        )
        self.tst = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=latent_dim,
                nhead=4,
                batch_first=True
            ),
            num_layers=2
        )
        self.correction_head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.Tanh()
        )
        self.static_density = nn.Linear(latent_dim, 1)
        self.dynamic_density = nn.Linear(latent_dim, 1)
        self.turb_density = nn.Linear(latent_dim, 1)
        self.risk_decoder = nn.Sequential(
            nn.Linear(latent_dim + 3, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, M, S, X_prev=None):
        B, K, Z, _ = M.shape

        M = self.motion_bn(
            M.permute(0, 3, 1, 2)
        ).permute(0, 2, 3, 1)

        S = self.sensor_bn(
            S.permute(0, 3, 1, 2)
        ).permute(0, 2, 3, 1)

        Tm = self.motion_enc(M)
        Ts = self.sensor_enc(S)

        alpha = self.gmf_gate(
            torch.cat([Tm, Ts], dim=-1)
        )

        Tf = alpha * Tm + (1 - alpha) * Ts

        E = self.zone_embedding(
            torch.arange(Z, device=M.device)
        ).view(1, 1, Z, -1)

        Zt = torch.cat(
            [Tf, E.expand(B, K, Z, -1)],
            dim=-1
        )

        I = self.lfie(
            Zt.transpose(1, 2).reshape(
                B * Z,
                self.latent_dim,
                K
            )
        ).mean(dim=-1).view(
            B, Z, self.latent_dim
        )

        if not self.training:
            U = torch.mean(
                torch.abs(Tm[:, -1] - Ts[:, -1]),
                dim=-1,
                keepdim=True
            )
            I = I * (1 - self.ats(U))

        X_seq = torch.cat([
            X_prev.unsqueeze(1)
            if X_prev is not None
            else torch.zeros(
                B,
                1,
                self.latent_dim,
                device=M.device
            ),
            I
        ], dim=1)

        X_out = self.tst(X_seq)

        X_curr = X_out[:, 0]
        Z_lat = X_out[:, 1:]

        X_curr = (
            X_curr
            + 0.1 * self.correction_head(X_curr)
        )

        ds = self.static_density(Z_lat)
        dd = self.dynamic_density(Z_lat)
        dc = self.turb_density(Z_lat)

        R = self.risk_decoder(
            torch.cat([Z_lat, ds, dd, dc], dim=-1)
        )

        return R, alpha, X_curr, (ds, dd, dc)


class LSTMClassifier(nn.Module):
    def __init__(
        self,
        input_dim=7,
        hidden_dim=64,
        num_layers=1
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.classifier = nn.Linear(
            hidden_dim,
            2
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])


class TCNClassifier(nn.Module):
    def __init__(
        self,
        input_dim=7,
        channels=(64, 64),
        kernel_size=3
    ):
        super().__init__()

        layers = []
        in_channels = input_dim

        for out_channels in channels:
            layers.extend([
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2
                ),
                nn.ReLU(),
                nn.BatchNorm1d(out_channels)
            ])
            in_channels = out_channels

        self.tcn = nn.Sequential(*layers)

        self.classifier = nn.Linear(
            channels[-1],
            2
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = x[:, :, -1]
        return self.classifier(x)


hydro_model = HydroFlowNet_XF()
lstm_model = LSTMClassifier()
tcn_model = TCNClassifier()

models = {
    "HydroFlowNet-XF": hydro_model,
    "LSTM": lstm_model,
    "TCN": tcn_model
}

print("=" * 75)
print("MODEL ARCHITECTURES")
print("=" * 75)

for name, model_obj in models.items():
    params = sum(
        p.numel()
        for p in model_obj.parameters()
        if p.requires_grad
    )
    print(f"{name:20s}: {params:,} trainable parameters")

print("=" * 75)
print("✔ Frozen architectures instantiated.")

MODEL ARCHITECTURES
HydroFlowNet-XF     : 2,733,206 trainable parameters
LSTM                : 18,818 trainable parameters
TCN                 : 14,146 trainable parameters
✔ Frozen architectures instantiated.


In [3]:
# FROZEN INPUT REPRESENTATIONS

BATCH_SIZE = 1

# HydroFlowNet-XF
# [batch, history, zones, motion_features]
M_input = torch.randn(
    BATCH_SIZE,
    HISTORY_K,
    NUM_ZONES,
    4
)

# [batch, history, zones, sensor_features]
S_input = torch.randn(
    BATCH_SIZE,
    HISTORY_K,
    NUM_ZONES,
    3
)

# LSTM / TCN
# [batch, history, 7 features]
TEMPORAL_INPUT = torch.randn(
    BATCH_SIZE,
    HISTORY_K,
    7
)

# Classical baseline representation
# [batch, 21 features]
CLASSICAL_INPUT = np.random.randn(
    BATCH_SIZE,
    21
).astype(np.float32)

print("=" * 75)
print("FROZEN INPUT DIMENSIONS")
print("=" * 75)
print("HydroFlowNet-XF M_t :", tuple(M_input.shape))
print("HydroFlowNet-XF S_t :", tuple(S_input.shape))
print("LSTM input           :", tuple(TEMPORAL_INPUT.shape))
print("TCN input            :", tuple(TEMPORAL_INPUT.shape))
print("Classical input      :", tuple(CLASSICAL_INPUT.shape))
print("=" * 75)

assert M_input.shape == (1, 3, 4, 4)
assert S_input.shape == (1, 3, 4, 3)
assert TEMPORAL_INPUT.shape == (1, 3, 7)
assert CLASSICAL_INPUT.shape == (1, 21)

print("✔ All frozen input dimensions verified.")

FROZEN INPUT DIMENSIONS
HydroFlowNet-XF M_t : (1, 3, 4, 4)
HydroFlowNet-XF S_t : (1, 3, 4, 3)
LSTM input           : (1, 3, 7)
TCN input            : (1, 3, 7)
Classical input      : (1, 21)
✔ All frozen input dimensions verified.


In [4]:
# CONTROLLED CPU INFERENCE LATENCY
# Batch size = 1

import gc

CPU_DEVICE = torch.device("cpu")

hydro_cpu = HydroFlowNet_XF().to(CPU_DEVICE).eval()
lstm_cpu = LSTMClassifier().to(CPU_DEVICE).eval()
tcn_cpu = TCNClassifier().to(CPU_DEVICE).eval()

M_cpu = M_input.to(CPU_DEVICE)
S_cpu = S_input.to(CPU_DEVICE)
TEMPORAL_cpu = TEMPORAL_INPUT.to(CPU_DEVICE)

def benchmark_torch_cpu(
    inference_fn,
    warmup=50,
    repeats=500
):
    with torch.no_grad():

        for _ in range(warmup):
            _ = inference_fn()

        times_ms = []

        for _ in range(repeats):
            start = time.perf_counter()
            _ = inference_fn()
            end = time.perf_counter()

            times_ms.append(
                (end - start) * 1000.0
            )

    times_ms = np.asarray(times_ms)

    return {
        "Mean_ms": float(times_ms.mean()),
        "Std_ms": float(times_ms.std(ddof=1)),
        "Median_ms": float(np.median(times_ms)),
        "Min_ms": float(times_ms.min()),
        "Max_ms": float(times_ms.max()),
        "Throughput_samples_per_s":
            float(1000.0 / times_ms.mean())
    }


benchmark_rows = []

torch_models = [
    (
        "HydroFlowNet-XF",
        hydro_cpu,
        lambda: hydro_cpu(M_cpu, S_cpu)
    ),
    (
        "LSTM",
        lstm_cpu,
        lambda: lstm_cpu(TEMPORAL_cpu)
    ),
    (
        "TCN",
        tcn_cpu,
        lambda: tcn_cpu(TEMPORAL_cpu)
    )
]

for name, model_obj, fn in torch_models:

    params = sum(
        p.numel()
        for p in model_obj.parameters()
        if p.requires_grad
    )

    result = benchmark_torch_cpu(fn)

    benchmark_rows.append({
        "Model": name,
        "Input": (
            "M:[1,3,4,4] + S:[1,3,4,3]"
            if name == "HydroFlowNet-XF"
            else "X:[1,3,7]"
        ),
        "Batch": 1,
        "Device": "CPU",
        "Parameters": params,
        **result
    })

latency_df = pd.DataFrame(benchmark_rows)

print("=" * 75)
print("CONTROLLED CPU INFERENCE LATENCY")
print("=" * 75)
display(
    latency_df.round(4)
)
print("=" * 75)

print("✔ CPU latency benchmark completed.")

CONTROLLED CPU INFERENCE LATENCY


,Model,Input,Batch,Device,Parameters,Mean_ms,Std_ms,Median_ms,Min_ms,Max_ms,Throughput_samples_per_s
0,HydroFlowNet-XF,"M:[1,3,4,4] + S:[1,3,4,3]",1,CPU,2733206,5.2177,3.4638,4.5712,2.7617,24.2568,191.6566
1,LSTM,"X:[1,3,7]",1,CPU,18818,0.2292,0.0240,0.2197,0.2086,0.3912,4362.7421
2,TCN,"X:[1,3,7]",1,CPU,14146,0.1632,0.0133,0.1570,0.1539,0.2735,6128.0650


✔ CPU latency benchmark completed.


In [5]:
# CLASSICAL BASELINE INFERENCE LATENCY
# Profiling only — no accuracy/result evaluation

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Synthetic profiling data with the exact frozen feature dimensionality.
# This data is used ONLY to create fitted estimator objects so that
# predict_proba() can be timed. No predictive performance is reported.

rng = np.random.RandomState(42)

X_profile_train = rng.randn(
    392 * NUM_ZONES,
    21
).astype(np.float32)

y_profile_train = rng.randint(
    0,
    2,
    size=392 * NUM_ZONES
)

X_profile_one = rng.randn(
    1,
    21
).astype(np.float32)

profile_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        max_iter=500,
        random_state=42
    )
}

for name, estimator in profile_models.items():
    print(f"Fitting profiling estimator: {name}")
    estimator.fit(
        X_profile_train,
        y_profile_train
    )

def benchmark_sklearn(
    inference_fn,
    warmup=50,
    repeats=500
):
    for _ in range(warmup):
        _ = inference_fn()

    times_ms = []

    for _ in range(repeats):
        start = time.perf_counter()
        _ = inference_fn()
        end = time.perf_counter()

        times_ms.append(
            (end - start) * 1000.0
        )

    times_ms = np.asarray(times_ms)

    return {
        "Mean_ms": float(times_ms.mean()),
        "Std_ms": float(times_ms.std(ddof=1)),
        "Median_ms": float(np.median(times_ms)),
        "Min_ms": float(times_ms.min()),
        "Max_ms": float(times_ms.max()),
        "Throughput_samples_per_s":
            float(1000.0 / times_ms.mean())
    }

classical_rows = []

for name, estimator in profile_models.items():

    result = benchmark_sklearn(
        lambda m=estimator:
            m.predict_proba(X_profile_one)
    )

    try:
        params = estimator.get_params()
    except Exception:
        params = {}

    classical_rows.append({
        "Model": name,
        "Input": "X:[1,21]",
        "Batch": 1,
        "Device": "CPU",
        "Parameters": np.nan,
        **result
    })

classical_latency_df = pd.DataFrame(
    classical_rows
)

print("=" * 75)
print("CLASSICAL BASELINE CPU LATENCY")
print("=" * 75)
display(
    classical_latency_df.round(4)
)
print("=" * 75)

print("✔ Classical baseline profiling completed.")

Fitting profiling estimator: Logistic Regression
Fitting profiling estimator: Random Forest
Fitting profiling estimator: MLP
CLASSICAL BASELINE CPU LATENCY


,Model,Input,Batch,Device,Parameters,Mean_ms,Std_ms,Median_ms,Min_ms,Max_ms,Throughput_samples_per_s
0,Logistic Regression,"X:[1,21]",1,CPU,NaN,0.5407,0.9395,0.2795,0.2035,6.3620,1849.5371
1,Random Forest,"X:[1,21]",1,CPU,NaN,77.2683,10.1153,77.6319,54.4340,98.5211,12.9419
2,MLP,"X:[1,21]",1,CPU,NaN,0.1601,0.0183,0.1560,0.1464,0.3321,6245.2488


✔ Classical baseline profiling completed.


In [6]:
# LATENCY TABLE + REPRODUCIBILITY ARTIFACTS

from pathlib import Path

OUTPUT_DIR = Path("hydroflownet_latency_benchmark")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

all_latency_df = pd.concat(
    [
        latency_df,
        classical_latency_df
    ],
    ignore_index=True
)

# Add benchmark metadata
all_latency_df["Benchmark"] = (
    "Batch-1 CPU inference profiling"
)

all_latency_df["Warmup_runs"] = 50
all_latency_df["Timed_runs"] = 500

# Reorder columns
all_latency_df = all_latency_df[
    [
        "Model",
        "Input",
        "Batch",
        "Device",
        "Parameters",
        "Mean_ms",
        "Std_ms",
        "Median_ms",
        "Min_ms",
        "Max_ms",
        "Throughput_samples_per_s",
        "Warmup_runs",
        "Timed_runs",
        "Benchmark"
    ]
]

print("=" * 75)
print("FINAL COMPUTATIONAL LATENCY RESULTS")
print("=" * 75)
display(
    all_latency_df.round(4)
)
print("=" * 75)

# Save latency results
latency_path = (
    OUTPUT_DIR /
    "controlled_cpu_latency_results.csv"
)

all_latency_df.to_csv(
    latency_path,
    index=False
)

# Save environment
environment_df = pd.DataFrame(
    [
        {
            "Parameter": key,
            "Value": value
        }
        for key, value in environment.items()
    ]
)

environment_path = (
    OUTPUT_DIR /
    "computational_environment.csv"
)

environment_df.to_csv(
    environment_path,
    index=False
)

# Save benchmark configuration
benchmark_config = pd.DataFrame([
    {
        "Parameter": "Batch size",
        "Value": 1
    },
    {
        "Parameter": "Warm-up runs",
        "Value": 50
    },
    {
        "Parameter": "Timed runs",
        "Value": 500
    },
    {
        "Parameter": "HydroFlowNet-XF input",
        "Value": "M:[1,3,4,4] + S:[1,3,4,3]"
    },
    {
        "Parameter": "LSTM input",
        "Value": "[1,3,7]"
    },
    {
        "Parameter": "TCN input",
        "Value": "[1,3,7]"
    },
    {
        "Parameter": "Classical baseline input",
        "Value": "[1,21]"
    },
    {
        "Parameter": "HydroFlowNet-XF training",
        "Value": "Not performed"
    },
    {
        "Parameter": "Accuracy evaluation",
        "Value": "Not performed"
    },
    {
        "Parameter": "Purpose",
        "Value": "Inference latency profiling only"
    }
])

config_path = (
    OUTPUT_DIR /
    "latency_benchmark_configuration.csv"
)

benchmark_config.to_csv(
    config_path,
    index=False
)

print("✔ Saved:")
print("  ", latency_path)
print("  ", environment_path)
print("  ", config_path)

print("=" * 75)
print("IMPORTANT")
print("This notebook performs NO model training.")
print("No manuscript accuracy/F1/AUC results are generated here.")
print("The frozen experimental notebook remains unchanged.")
print("=" * 75)

FINAL COMPUTATIONAL LATENCY RESULTS


,Model,Input,Batch,Device,Parameters,Mean_ms,Std_ms,Median_ms,Min_ms,Max_ms,Throughput_samples_per_s,Warmup_runs,Timed_runs,Benchmark
0,HydroFlowNet-XF,"M:[1,3,4,4] + S:[1,3,4,3]",1,CPU,2733206.0,5.2177,3.4638,4.5712,2.7617,24.2568,191.6566,50,500,Batch-1 CPU inference profiling
1,LSTM,"X:[1,3,7]",1,CPU,18818.0,0.2292,0.0240,0.2197,0.2086,0.3912,4362.7421,50,500,Batch-1 CPU inference profiling
2,TCN,"X:[1,3,7]",1,CPU,14146.0,0.1632,0.0133,0.1570,0.1539,0.2735,6128.0650,50,500,Batch-1 CPU inference profiling
3,Logistic Regression,"X:[1,21]",1,CPU,NaN,0.5407,0.9395,0.2795,0.2035,6.3620,1849.5371,50,500,Batch-1 CPU inference profiling
4,Random Forest,"X:[1,21]",1,CPU,NaN,77.2683,10.1153,77.6319,54.4340,98.5211,12.9419,50,500,Batch-1 CPU inference profiling
5,MLP,"X:[1,21]",1,CPU,NaN,0.1601,0.0183,0.1560,0.1464,0.3321,6245.2488,50,500,Batch-1 CPU inference profiling


✔ Saved:
   hydroflownet_latency_benchmark/controlled_cpu_latency_results.csv
   hydroflownet_latency_benchmark/computational_environment.csv
   hydroflownet_latency_benchmark/latency_benchmark_configuration.csv
IMPORTANT
This notebook performs NO model training.
No manuscript accuracy/F1/AUC results are generated here.
The frozen experimental notebook remains unchanged.
